# NLP - Task 5
## Noor Zahid (SP24-BDS-059)
## Tokenization, Perplexity, Smoothning Techniques

### Task 1

In [1]:
from collections import Counter
from itertools import permutations
from math import log, exp, inf
from itertools import islice

train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]

test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]

print("Training sentences:", len(train))
print("Testing sentences:", len(test))

Training sentences: 4
Testing sentences: 2


In [4]:
def tokenize(sentence):
    sentence = sentence.lower()
    words = sentence.split(" ")
    return ["<s>"] + words + ["</s>"]


print("Training tokens:")
for sentence in train:
    print(tokenize(sentence))

print("\nTesting tokens:")
for sentence in test:
    print(tokenize(sentence))

Training tokens:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']

Testing tokens:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']


### With and without markers

In [5]:
train_tokens = [tokenize(sentence) for sentence in train]

tokens_with_markers = sum(len(tokens) for tokens in train_tokens)
tokens_without_markers = sum(
    len(tokens) - 2 for tokens in train_tokens
)

print("Tokens with markers:", tokens_with_markers)
print("Tokens without markers:", tokens_without_markers)

Tokens with markers: 30
Tokens without markers: 22


### Build the vocabulary

In [6]:
def build_vocab(sentences):
    vocab = set()

    for sentence in sentences:
        words = sentence.lower().split(" ")
        vocab.update(words)

    vocab.add("</s>")
    vocab.discard("<s>")

    return vocab


V = build_vocab(train)

print("Vocabulary V:", V)
print("\nSorted vocabulary:")
print(sorted(V))
print("\nVocabulary size:", len(V))

Vocabulary V: {'rug', 'cat', 'the', 'mat', 'ate', 'bone', 'dog', '</s>', 'fish', 'on', 'sat'}

Sorted vocabulary:
['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']

Vocabulary size: 11


### Task 2 — Generating N-grams

In [7]:
def get_ngrams(tokens, n):
    if n <= 0:
        raise ValueError("n must be greater than 0")

    return [
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    ]


first_sentence = train_tokens[0]

print("Tokens:", first_sentence)
print("\nUnigrams:", get_ngrams(first_sentence, 1))
print("\nBigrams:", get_ngrams(first_sentence, 2))
print("\nTrigrams:", get_ngrams(first_sentence, 3))

Tokens: ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']

Unigrams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]

Bigrams: [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]

Trigrams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]


### Verify the N-gram formula 

In [8]:
for sentence_number, tokens in enumerate(train_tokens, start=1):
    m = len(tokens)

    for n in (1, 2, 3):
        ngrams = get_ngrams(tokens, n)
        expected = max(0, m - n + 1)

        print(
            f"Sentence {sentence_number}, n={n}: "
            f"actual={len(ngrams)}, expected={expected}"
        )

        assert len(ngrams) == expected

print("\nAll N-gram checks passed!")

Sentence 1, n=1: actual=8, expected=8
Sentence 1, n=2: actual=7, expected=7
Sentence 1, n=3: actual=6, expected=6
Sentence 2, n=1: actual=7, expected=7
Sentence 2, n=2: actual=6, expected=6
Sentence 2, n=3: actual=5, expected=5
Sentence 3, n=1: actual=8, expected=8
Sentence 3, n=2: actual=7, expected=7
Sentence 3, n=3: actual=6, expected=6
Sentence 4, n=1: actual=7, expected=7
Sentence 4, n=2: actual=6, expected=6
Sentence 4, n=3: actual=5, expected=5

All N-gram checks passed!


### When n is larger than the sentence

In [10]:
short_tokens = ["the", "cat"]

print("Tokens:", short_tokens)
print("Bigrams:", get_ngrams(short_tokens, 2))
print("Trigrams:", get_ngrams(short_tokens, 3))
print("Four-grams:", get_ngrams(short_tokens, 4))

Tokens: ['the', 'cat']
Bigrams: [('the', 'cat')]
Trigrams: []
Four-grams: []


### Task 3 — Counting N-grams

In [11]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update(
        token for token in tokens if token != "<s>"
    )
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))


def print_count_table(counts, name):
    print(f"\n{name} counts:")

    for ngram, count in sorted(
        counts.items(),
        key=lambda item: (-item[1], item[0])
    ):
        print(f"{ngram}: {count}")


print_count_table(unigram_counts, "Unigram")
print_count_table(bigram_counts, "Bigram")
print_count_table(trigram_counts, "Trigram")


Unigram counts:
the: 8
</s>: 4
ate: 2
cat: 2
dog: 2
on: 2
sat: 2
bone: 1
fish: 1
mat: 1
rug: 1

Bigram counts:
('<s>', 'the'): 4
('ate', 'the'): 2
('on', 'the'): 2
('sat', 'on'): 2
('the', 'cat'): 2
('the', 'dog'): 2
('bone', '</s>'): 1
('cat', 'ate'): 1
('cat', 'sat'): 1
('dog', 'ate'): 1
('dog', 'sat'): 1
('fish', '</s>'): 1
('mat', '</s>'): 1
('rug', '</s>'): 1
('the', 'bone'): 1
('the', 'fish'): 1
('the', 'mat'): 1
('the', 'rug'): 1

Trigram counts:
('<s>', 'the', 'cat'): 2
('<s>', 'the', 'dog'): 2
('sat', 'on', 'the'): 2
('ate', 'the', 'bone'): 1
('ate', 'the', 'fish'): 1
('cat', 'ate', 'the'): 1
('cat', 'sat', 'on'): 1
('dog', 'ate', 'the'): 1
('dog', 'sat', 'on'): 1
('on', 'the', 'mat'): 1
('on', 'the', 'rug'): 1
('the', 'bone', '</s>'): 1
('the', 'cat', 'ate'): 1
('the', 'cat', 'sat'): 1
('the', 'dog', 'ate'): 1
('the', 'dog', 'sat'): 1
('the', 'fish', '</s>'): 1
('the', 'mat', '</s>'): 1
('the', 'rug', '</s>'): 1


### Five most common bigrams and distinct counts

In [12]:
print("Five most common bigrams:")

for bigram, count in sorted(
    bigram_counts.items(),
    key=lambda item: (-item[1], item[0])
)[:5]:
    print(bigram, "->", count)

print("\nDistinct unigrams:", len(unigram_counts))
print("Distinct bigrams:", len(bigram_counts))
print("Distinct trigrams:", len(trigram_counts))

Five most common bigrams:
('<s>', 'the') -> 4
('ate', 'the') -> 2
('on', 'the') -> 2
('sat', 'on') -> 2
('the', 'cat') -> 2

Distinct unigrams: 11
Distinct bigrams: 18
Distinct trigrams: 19


### Count seen and unseen N-grams

In [13]:
def count_of(ngram):
    if isinstance(ngram, str):
        return unigram_counts.get(ngram, 0)

    if len(ngram) == 2:
        return bigram_counts.get(tuple(ngram), 0)

    if len(ngram) == 3:
        return trigram_counts.get(tuple(ngram), 0)

    return unigram_counts.get(ngram[0], 0)


print("Seen bigram:", count_of(("the", "cat")))
print("Unseen bigram:", count_of(("cat", "bone")))

Seen bigram: 2
Unseen bigram: 0


### Task 4 — Unigram model

In [14]:
N = sum(unigram_counts.values())


def unigram_prob(w):
    return unigram_counts.get(w, 0) / N


print(f"{'Word':<12} {'Count':<8} {'Probability':<12}")

for word in sorted(V):
    print(
        f"{word:<12} "
        f"{unigram_counts[word]:<8} "
        f"{unigram_prob(word):.4f}"
    )

total_probability = sum(unigram_prob(w) for w in V)

print("\nSum of probabilities:", round(total_probability, 4))
assert abs(total_probability - 1.0) < 1e-10

Word         Count    Probability 
</s>         4        0.1538
ate          2        0.0769
bone         1        0.0385
cat          2        0.0769
dog          2        0.0769
fish         1        0.0385
mat          1        0.0385
on           2        0.0769
rug          1        0.0385
sat          2        0.0769
the          8        0.3077

Sum of probabilities: 1.0


### Probability of each test sentence

In [15]:
def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence)

    probability = 1.0

    for word in tokens[1:]:
        probability *= unigram_prob(word)

    return probability


for sentence in test:
    print(sentence)
    print("Probability:", unigram_sentence_prob(sentence))

the cat sat on the rug
Probability: 2.549861058867735e-07
the fish sat on the mat
Probability: 1.2749305294338675e-07


### Demonstrate that word order does not matter

In [16]:
first_test_words = test[0].split()

unique_orders = sorted(set(permutations(first_test_words)))

print("Original sentence:", first_test_words)
print("\nFirst five word orders:")

for order in unique_orders[:5]:
    sentence = " ".join(order)
    print(sentence, "->", unigram_sentence_prob(sentence))

probabilities = {
    unigram_sentence_prob(" ".join(order))
    for order in unique_orders
}

print("\nNumber of different probabilities:", len(probabilities))
assert len(probabilities) == 1

Original sentence: ['the', 'cat', 'sat', 'on', 'the', 'rug']

First five word orders:
cat on rug sat the the -> 2.549861058867735e-07
cat on rug the sat the -> 2.549861058867735e-07
cat on rug the the sat -> 2.549861058867735e-07
cat on sat rug the the -> 2.549861058867735e-07
cat on sat the rug the -> 2.549861058867735e-07

Number of different probabilities: 1


### Task 5 — Bigram model (MLE)

In [17]:
def history_count(prev):
    return sum(
        count for (first, second), count in bigram_counts.items()
        if first == prev
    )


def bigram_prob(prev, w):
    denominator = history_count(prev)

    if denominator == 0:
        return 0.0

    return bigram_counts.get((prev, w), 0) / denominator


print("P(cat | the) =", bigram_prob("the", "cat"))
print("P(sat | fish) =", bigram_prob("fish", "sat"))
print("P(bone | cat) =", bigram_prob("cat", "bone"))

P(cat | the) = 0.25
P(sat | fish) = 0.0
P(bone | cat) = 0.0


### Next-word distribution

In [18]:
def next_word_distribution(prev):
    return {
        word: bigram_prob(prev, word)
        for word in sorted(V)
    }


distribution = next_word_distribution("the")

print("Next-word distribution after 'the':")

for word, probability in distribution.items():
    print(f"{word:<12} {probability:.4f}")

print("\nSum:", sum(distribution.values()))

assert abs(sum(distribution.values()) - 1.0) < 1e-10

Next-word distribution after 'the':
</s>         0.0000
ate          0.0000
bone         0.1250
cat          0.2500
dog          0.2500
fish         0.1250
mat          0.1250
on           0.0000
rug          0.1250
sat          0.0000
the          0.0000

Sum: 1.0


### Bigram sentence probability

In [19]:
def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    probability = 1.0

    for i in range(1, len(tokens)):
        probability *= bigram_prob(tokens[i - 1], tokens[i])

    return probability


for sentence in test:
    print(sentence)
    print("Bigram probability:", bigram_sentence_prob(sentence))

the cat sat on the rug
Bigram probability: 0.015625
the fish sat on the mat
Bigram probability: 0.0


### Predict the next word

In [20]:
def predict_next(prev):
    distribution = next_word_distribution(prev)

    return min(
        distribution,
        key=lambda word: (-distribution[word], word)
    )


for word in ["the", "cat", "dog", "fish", "sat", "on"]:
    print(f"After '{word}' -> '{predict_next(word)}'")

After 'the' -> 'cat'
After 'cat' -> 'ate'
After 'dog' -> 'ate'
After 'fish' -> '</s>'
After 'sat' -> 'on'
After 'on' -> 'the'


### Task 6 — Trigram model

In [21]:
def trigram_prob(w1, w2, w3):
    denominator = bigram_counts.get((w1, w2), 0)

    if denominator == 0:
        return 0.0

    return trigram_counts.get((w1, w2, w3), 0) / denominator


print("P(sat | the, cat) =",
      trigram_prob("the", "cat", "sat"))

print("P(bone | the, cat) =",
      trigram_prob("the", "cat", "bone"))

P(sat | the, cat) = 0.5
P(bone | the, cat) = 0.0


### Trigram sentence probability

In [22]:
def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)

    if len(tokens) < 2:
        return 0.0

    probability = bigram_prob(tokens[0], tokens[1])

    for i in range(2, len(tokens)):
        probability *= trigram_prob(
            tokens[i - 2],
            tokens[i - 1],
            tokens[i]
        )

    return probability


for sentence in test:
    print(sentence)
    print("Trigram probability:", trigram_sentence_prob(sentence))

the cat sat on the rug
Trigram probability: 0.125
the fish sat on the mat
Trigram probability: 0.0


### Distinct N-gram counts

In [23]:
print(f"{'N-gram':<12} {'Distinct types':<15}")
print("-" * 27)
print(f"{'Unigrams':<12} {len(unigram_counts):<15}")
print(f"{'Bigrams':<12} {len(bigram_counts):<15}")
print(f"{'Trigrams':<12} {len(trigram_counts):<15}")

N-gram       Distinct types 
---------------------------
Unigrams     11             
Bigrams      18             
Trigrams     19             


### Find unseen bigrams and trigrams

In [24]:
def unseen_bigrams(sentence):
    tokens = tokenize(sentence)

    return [
        bg for bg in get_ngrams(tokens, 2)
        if bigram_counts.get(bg, 0) == 0
    ]


def unseen_trigrams(sentence):
    tokens = tokenize(sentence)

    return [
        tg for tg in get_ngrams(tokens, 3)
        if trigram_counts.get(tg, 0) == 0
    ]


for sentence in test:
    unseen_bg = unseen_bigrams(sentence)
    unseen_tg = unseen_trigrams(sentence)

    print("\nSentence:", sentence)
    print("Unseen bigrams:", unseen_bg)
    print("Unseen trigrams:", unseen_tg)

    if len(unseen_bg) > len(unseen_tg):
        print("More unseen N-grams: bigrams")
    elif len(unseen_tg) > len(unseen_bg):
        print("More unseen N-grams: trigrams")
    else:
        print("Both have the same number of unseen N-grams")


Sentence: the cat sat on the rug
Unseen bigrams: []
Unseen trigrams: []
Both have the same number of unseen N-grams

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
More unseen N-grams: trigrams


### Task 7 — Zero-probability problem

In [25]:
for sentence in test:
    print("\nSentence:", sentence)
    print("Unseen bigrams:", unseen_bigrams(sentence))
    print("Bigram MLE probability:", bigram_sentence_prob(sentence))


Sentence: the cat sat on the rug
Unseen bigrams: []
Bigram MLE probability: 0.015625

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Bigram MLE probability: 0.0


### count word-to-word count

In [26]:
for sentence in test:
    print("\nSentence:", sentence)
    print("Unseen bigrams:", unseen_bigrams(sentence))
    print("Bigram MLE probability:", bigram_sentence_prob(sentence))


Sentence: the cat sat on the rug
Unseen bigrams: []
Bigram MLE probability: 0.015625

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Bigram MLE probability: 0.0


### Safe log 


In [27]:
def safe_log(p):
    if p == 0:
        return -inf

    return log(p)


second_test_probability = bigram_sentence_prob(test[1])

print("Second test sentence:", test[1])
print("Probability:", second_test_probability)
print("Log probability:", safe_log(second_test_probability))

Second test sentence: the fish sat on the mat
Probability: 0.0
Log probability: -inf


### Task 8 — Smoothing

In [28]:
def bigram_prob_addone(prev, w):
    count = bigram_counts.get((prev, w), 0)
    denominator = history_count(prev) + len(V)

    return (count + 1) / denominator


print(f"{'Probability':<20} {'MLE':<12} {'Add-one':<12}")
print("-" * 44)

comparisons = [
    ("P(sat | fish)", "fish", "sat"),
    ("P(cat | the)", "the", "cat")
]

for label, prev, word in comparisons:
    print(
        f"{label:<20} "
        f"{bigram_prob(prev, word):<12.4f} "
        f"{bigram_prob_addone(prev, word):<12.4f}"
    )


Probability          MLE          Add-one     
--------------------------------------------
P(sat | fish)        0.0000       0.0833      
P(cat | the)         0.2500       0.1579      


### Verify all smoothed distributions sum to 1

In [29]:
previous_words = ["<s>"] + sorted(V)

for prev in previous_words:
    total = sum(
        bigram_prob_addone(prev, word)
        for word in V
    )

    assert abs(total - 1.0) < 1e-10, (
        f"Probabilities do not sum to 1 for {prev}"
    )

print("All add-one distributions sum to 1!")

All add-one distributions sum to 1!


### Implement add-k smoothing

In [30]:
def bigram_prob_addk(prev, w, k):
    if k <= 0:
        raise ValueError("k must be greater than 0")

    count = bigram_counts.get((prev, w), 0)
    denominator = history_count(prev) + k * len(V)

    return (count + k) / denominator


print(f"{'k':<10} {'P(sat | fish)':<15}")
print("-" * 25)

for k in [1, 0.5, 0.1, 0.01]:
    probability = bigram_prob_addk("fish", "sat", k)
    print(f"{k:<10} {probability:.4f}")

k          P(sat | fish)  
-------------------------
1          0.0833
0.5        0.0769
0.1        0.0476
0.01       0.0090


### Task 9 — Perplexity and model comparison

### probability ftns for perplexity

In [31]:
def bigram_probs_for_tokens(tokens):
    return [
        bigram_prob(tokens[i - 1], tokens[i])
        for i in range(1, len(tokens))
    ]


def addone_bigram_probs_for_tokens(tokens):
    return [
        bigram_prob_addone(tokens[i - 1], tokens[i])
        for i in range(1, len(tokens))
    ]


def addk_bigram_probs_for_tokens(tokens, k):
    return [
        bigram_prob_addk(tokens[i - 1], tokens[i], k)
        for i in range(1, len(tokens))
    ]


def unigram_prob_addone(word):
    return (
        unigram_counts.get(word, 0) + 1
    ) / (N + len(V))


def addone_unigram_probs_for_tokens(tokens):
    return [
        unigram_prob_addone(word)
        for word in tokens[1:]
    ]

In [32]:
def perplexity(sentences, prob_fn):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        tokens = tokenize(sentence)
        probabilities = prob_fn(tokens)

        for probability in probabilities:
            if probability == 0:
                return inf

            total_log_probability += log(probability)
            total_tokens += 1

    if total_tokens == 0:
        return inf

    return exp(-total_log_probability / total_tokens)

In [33]:
models = [
    ("Bigram MLE",
     bigram_probs_for_tokens),

    ("Bigram add-one",
     addone_bigram_probs_for_tokens),

    ("Unigram add-one",
     addone_unigram_probs_for_tokens)
]

print(f"{'Model':<22} {'Test perplexity':<18}")
print("-" * 40)

for name, prob_fn in models:
    score = perplexity(test, prob_fn)
    print(f"{name:<22} {score:.4f}")

Model                  Test perplexity   
----------------------------------------
Bigram MLE             inf
Bigram add-one         5.7969
Unigram add-one        9.1370


### Compare add-k values on the test set

In [34]:
k_values = [1, 0.5, 0.1, 0.01]
addk_results = {}

print(f"{'k':<10} {'Test perplexity':<18}")
print("-" * 28)

for k in k_values:
    prob_fn = lambda tokens, k=k: addk_bigram_probs_for_tokens(
        tokens, k
    )

    score = perplexity(test, prob_fn)
    addk_results[k] = score

    print(f"{k:<10} {score:.4f}")

best_test_k = min(addk_results, key=addk_results.get)

print("\nBest k directly on test set:", best_test_k)
print("Best test-set perplexity:", addk_results[best_test_k])

k          Test perplexity   
----------------------------
1          5.7969
0.5        4.5703
0.1        2.9559
0.01       2.6272

Best k directly on test set: 0.01
Best test-set perplexity: 2.6272203528005287


### Validation using only the first three training sentences

In [35]:
def build_counts_from_sentences(sentences):
    unigram = Counter()
    bigram = Counter()
    trigram = Counter()

    for sentence in sentences:
        tokens = tokenize(sentence)

        unigram.update(
            word for word in tokens if word != "<s>"
        )
        bigram.update(get_ngrams(tokens, 2))
        trigram.update(get_ngrams(tokens, 3))

    return unigram, bigram, trigram


# Save the full-training counts so we can restore them later.
full_unigram_counts = unigram_counts
full_bigram_counts = bigram_counts
full_trigram_counts = trigram_counts
full_N = N

# Train on only the first three sentences.
unigram_counts, bigram_counts, trigram_counts = (
    build_counts_from_sentences(train[:3])
)

N = sum(unigram_counts.values())

validation_sentence = [train[3]]

validation_results = {}

print("Validation sentence:", train[3])
print("\nValidation perplexity for each k:")

for k in k_values:
    prob_fn = lambda tokens, k=k: addk_bigram_probs_for_tokens(
        tokens, k
    )

    score = perplexity(validation_sentence, prob_fn)
    validation_results[k] = score

    print(f"k={k:<5} perplexity={score:.4f}")

best_validation_k = min(
    validation_results,
    key=validation_results.get
)

print("\nBest k on validation data:", best_validation_k)

Validation sentence: the dog ate the bone

Validation perplexity for each k:
k=1     perplexity=8.5857
k=0.5   perplexity=8.0163
k=0.1   perplexity=8.0260
k=0.01  perplexity=13.1304

Best k on validation data: 0.5


### Evaluate the selected k on the test set

In [36]:
test_score_with_validation_k = perplexity(
    test,
    lambda tokens: addk_bigram_probs_for_tokens(
        tokens, best_validation_k
    )
)

print("Best k selected on validation:", best_validation_k)
print(
    "Test perplexity using validation-selected k:",
    round(test_score_with_validation_k, 4)
)

print(
    "\nBest k selected directly on test:",
    best_test_k
)

print(
    "Best perplexity found directly on test:",
    round(addk_results[best_test_k], 4)
)

# Restore the counts from all four training sentences.
unigram_counts = full_unigram_counts
bigram_counts = full_bigram_counts
trigram_counts = full_trigram_counts
N = full_N

Best k selected on validation: 0.5
Test perplexity using validation-selected k: 4.4539

Best k selected directly on test: 0.01
Best perplexity found directly on test: 2.6272
